In [1]:
# 01. EXTRAÇÃO FRAME A FRAME E ERSP (N170)

import os
import glob
import mne
import warnings
import numpy as np
import pandas as pd
from scipy.stats import entropy

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("Iniciando Extração Dinâmica (Frame a Frame) + ERSP...")
print("="*80)

# 1. Diretórios (Mantenha igual ao seu padrão)
DIR_EPOCHS = '../data/processed/epochs/'
DIR_REPORTS = '../reports/'
os.makedirs(DIR_REPORTS, exist_ok=True)
ARQUIVO_SAIDA = os.path.join(DIR_REPORTS, 'features_frame_a_frame_ERSP.csv')

# 2. Parâmetros Base
BANDAS_FREQ = {'Theta': (4.0, 8.0), 'Alpha': (8.0, 13.0), 'Beta': (13.0, 30.0), 'Gamma': (30.0, 40.0)}
CANAIS_PSD = ['Fp1', 'Fp2', 'F7', 'F3', 'F4', 'F8', 'T3', 'C3', 'C4', 'T4', 'T5', 'P3', 'P4', 'T6', 'O1', 'O2']
PARES_ASSIMETRIA = [('F4', 'F3'), ('C4', 'C3'), ('P4', 'P3'), ('O2', 'O1')]
PARES_CONECTIVIDADE = [('F3', 'F4'), ('C3', 'C4'), ('P3', 'P4'), ('O1', 'O2'), ('F3', 'O1'), ('F4', 'O2')]

# 3. Novos Parâmetros para o ERSP (N170 na Frequência)
CANAIS_ERSP = ['O1', 'O2', 'T5', 'T6'] # Expandido para incluir os temporais
JANELA_ERSP = (0.150, 0.200) # Janela estrita de 150ms a 200ms
FREQS_ERSP = np.arange(4, 13, 1) # Pesquisando energia de 4 a 12 Hz (Theta e Alpha)

arquivos_epo = sorted(glob.glob(os.path.join(DIR_EPOCHS, '*-epo.fif')))
todas_features = []

# 4. Varredura dos Pacientes
for caminho_arquivo in arquivos_epo:
    nome_arquivo = os.path.basename(caminho_arquivo)
    sujeito_id = nome_arquivo.split('-epo')[0]
    grupo = 'TEA' if 'TEA' in sujeito_id else 'Control'
    
    epochs = mne.read_epochs(caminho_arquivo, preload=True, verbose=False)
    
    for condicao in epochs.event_id.keys():
        epochs_condicao = epochs[condicao]
        n_epochs = len(epochs_condicao)
        
        if n_epochs == 0:
            continue

        # A. Cálculos Globais (Computa tudo de uma vez para economizar processamento)
        n_tempos = epochs_condicao.get_data().shape[2]
        n_fft = min(256, n_tempos) 
        
        # PSD (Welch) para todas as épocas de uma vez
        psd_data = epochs_condicao.compute_psd(method='welch', fmin=4.0, fmax=40.0, 
                                               n_fft=n_fft, n_overlap=n_fft//2, verbose=False)
        psds, freqs = psd_data.get_data(return_freqs=True) # Shape: (epochs, canais, frequencias)
        
        # B. TFR (Morlet Wavelets) para o ERSP (Apenas se for tarefa de Faces)
        nome_condicao_limpo = condicao
        if condicao in ['FF', 'F']: nome_condicao_limpo = 'Face Feliz'
        elif condicao in ['FN', 'N']: nome_condicao_limpo = 'Face Neutra'
        elif condicao in ['FR', 'R']: nome_condicao_limpo = 'Face Raiva'
        
        tipo_tarefa = 'Task' if 'Face' in nome_condicao_limpo else 'Resting'
        
        tfr_data = None
        if tipo_tarefa == 'Task':
            # Calcula a wavelet época a época (average=False)
            tfr = mne.time_frequency.tfr_morlet(epochs_condicao, freqs=FREQS_ERSP, 
                                                n_cycles=FREQS_ERSP/2.0, return_itc=False, 
                                                average=False, verbose=False)
            tfr_data = tfr.data # Shape: (epochs, canais, freqs, tempos)
            tfr_times = tfr.times

        # C. O NOVO LAÇO: Extração Frame a Frame
        # Em vez de tirar a média, vamos varrer cada época (frame) individualmente
        dados_brutos_epochs = epochs_condicao.get_data()
        
        for epoch_idx in range(n_epochs):
            # Dicionário base para esta linha específica (este frame)
            features_linha = {
                'ID': sujeito_id,
                'Grupo': grupo,
                'Condicao': nome_condicao_limpo,
                'Tipo': tipo_tarefa,
                'Frame_Num': epoch_idx + 1  # Guarda se é a foto 1, 2, 3... até 30
            }
            
            # Pega o PSD apenas desta época
            psd_epoca = psds[epoch_idx, :, :]
            
            # Extração de Potência e Entropia para ESTE frame
            potencia_absoluta = {}
            for ch_idx, canal in enumerate(epochs_condicao.ch_names):
                if canal not in CANAIS_PSD: continue
                potencia_total_canal = 0
                
                # Potência Absoluta
                for banda, (fmin, fmax) in BANDAS_FREQ.items():
                    idx_banda = np.logical_and(freqs >= fmin, freqs <= fmax)
                    potencia = np.sum(psd_epoca[ch_idx, idx_banda])
                    potencia_absoluta[f"{banda}_{canal}"] = potencia
                    potencia_total_canal += potencia
                
                # Potência Relativa
                for banda in BANDAS_FREQ.keys():
                    abs_val = potencia_absoluta[f"{banda}_{canal}"]
                    rel_val = abs_val / potencia_total_canal if potencia_total_canal > 0 else 0
                    features_linha[f"{banda}Rel_{canal}"] = rel_val
                
                # Razão Theta/Beta
                theta_val = potencia_absoluta[f"Theta_{canal}"]
                beta_val = potencia_absoluta[f"Beta_{canal}"]
                features_linha[f"TBR_{canal}"] = theta_val / beta_val if beta_val > 0 else 0
                
                # Entropia Espectral
                psd_norm = psd_epoca[ch_idx, :] / np.sum(psd_epoca[ch_idx, :])
                features_linha[f"Entropia_{canal}"] = entropy(psd_norm)

            # Assimetria para ESTE frame
            for banda in BANDAS_FREQ.keys():
                for ch_dir, ch_esq in PARES_ASSIMETRIA:
                    chave_dir = f"{banda}Rel_{ch_dir}"
                    chave_esq = f"{banda}Rel_{ch_esq}"
                    if chave_dir in features_linha and chave_esq in features_linha:
                        nome_regiao = ch_dir[0] 
                        features_linha[f"Asym_{banda}_{nome_regiao}"] = features_linha[chave_dir] - features_linha[chave_esq]

            # Conectividade para ESTE frame
            dado_epoca_atual = dados_brutos_epochs[epoch_idx, :, :]
            for ch1, ch2 in PARES_CONECTIVIDADE:
                if ch1 in epochs_condicao.ch_names and ch2 in epochs_condicao.ch_names:
                    idx1 = epochs_condicao.ch_names.index(ch1)
                    idx2 = epochs_condicao.ch_names.index(ch2)
                    corr = np.corrcoef(dado_epoca_atual[idx1, :], dado_epoca_atual[idx2, :])[0, 1]
                    features_linha[f"Conn_{ch1}_{ch2}"] = corr

            # Nova Feature: ERSP para ESTE frame
            if tipo_tarefa == 'Task' and tfr_data is not None:
                # Isola os índices de tempo entre 150 e 200ms
                idx_tempo_ersp = np.logical_and(tfr_times >= JANELA_ERSP[0], tfr_times <= JANELA_ERSP[1])
                
                for canal in CANAIS_ERSP:
                    if canal in epochs_condicao.ch_names:
                        ch_idx = epochs_condicao.ch_names.index(canal)
                        # Tira a média da energia (power) apenas na janela de 150-200ms para as frequências selecionadas
                        energia_n170 = np.mean(tfr_data[epoch_idx, ch_idx, :, idx_tempo_ersp])
                        features_linha[f"ERSP_N170_{canal}"] = energia_n170

            todas_features.append(features_linha)

# 5. Exportação
df_features = pd.DataFrame(todas_features)
df_features.fillna(0, inplace=True)
df_features.to_csv(ARQUIVO_SAIDA, index=False)

print(f"Matriz de características DINÂMICA gerada com sucesso.")
print(f"Dimensões do dataset de saída: {df_features.shape[0]} amostras (frames) x {df_features.shape[1]} atributos.")
print("="*80)


Iniciando Extração Dinâmica (Frame a Frame) + ERSP...
Matriz de características DINÂMICA gerada com sucesso.
Dimensões do dataset de saída: 8375 amostras (frames) x 127 atributos.


In [2]:
# 02. PIPELINE DE CLASSIFICAÇÃO FRAME A FRAME (TIME-SERIES)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

print("\n" + "="*80)
print("Iniciando Análise Temporal: XGBoost Frame a Frame (Face Feliz)")
print("="*80)

# 1. Carregamento e Preparação dos Dados
df = pd.read_csv('../reports/features_frame_a_frame_ERSP.csv')

# Filtrar apenas para a condição alvo (Face Feliz)
df_feliz = df[df['Condicao'] == 'Face Feliz'].copy()

# Mapeamento do Target (TEA = 1, Controle = 0)
df_feliz['Target'] = df_feliz['Grupo'].map({'TEA': 1, 'Control': 0})

# Separar metadados e features
colunas_ignorar = ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']
features_cols = [c for c in df_feliz.columns if c not in colunas_ignorar]

# 2. Definição do Pipeline de Machine Learning
# Definindo o scale_pos_weight dinamicamente com base na proporção do grupo
n_controles = len(df_feliz[df_feliz['Target'] == 0]['ID'].unique())
n_tea = len(df_feliz[df_feliz['Target'] == 1]['ID'].unique())
peso_classes = n_controles / n_tea if n_tea > 0 else 1.0

logo = LeaveOneGroupOut()
frames = sorted(df_feliz['Frame_Num'].unique())
acuracias_por_frame = []

# 3. O Loop Temporal (Máquina do Tempo)
for frame in frames:
    # Isola os dados EXATAMENTE deste frame
    df_frame = df_feliz[df_feliz['Frame_Num'] == frame]
    
    # Se algum paciente perdeu este frame (ex: autoreject), garantimos que os dados batem
    X = df_frame[features_cols].values
    y = df_frame['Target'].values
    grupos_pacientes = df_frame['ID'].values 
    
    previsoes = []
    valores_reais = []
    
    # Validação Cruzada Isolando o Paciente (Prevenção de Data Leakage)
    for train_index, test_index in logo.split(X, y, grupos_pacientes):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Etapa A: Padronização (Ajustada apenas no treino)
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Etapa B: Seleção de Features (ANOVA) - As 15 melhores para ESTE frame
        selector = SelectKBest(score_func=f_classif, k=15)
        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)
        
        # Etapa C: Otimização e Treinamento do Modelo
        modelo = XGBClassifier(
            n_estimators=100,
            max_depth=3,
            scale_pos_weight=peso_classes,
            eval_metric='logloss',
            random_state=42
        )
        modelo.fit(X_train_sel, y_train)
        
        # Etapa D: Previsão no Paciente Oculto
        y_pred = modelo.predict(X_test_sel)
        previsoes.extend(y_pred)
        valores_reais.extend(y_test)
        
    # Calcula a acurácia global deste frame específico
    acc_frame = accuracy_score(valores_reais, previsoes)
    acuracias_por_frame.append(acc_frame)
    print(f"Frame {int(frame):02d}/30 concluído | Acurácia: {acc_frame*100:.2f}%")

# 4. Plotagem da Curva de Atenção/Engajamento Temporal
plt.figure(figsize=(12, 6))
plt.plot(frames, acuracias_por_frame, marker='o', linestyle='-', color='#1f77b4', linewidth=2)
plt.axhline(y=0.8095, color='r', linestyle='--', label='Baseline Qualificação (80.95%)')
plt.axhline(y=0.50, color='gray', linestyle=':', label='Chance Nível (50%)')

plt.title('Dinâmica Temporal de Classificação (XGBoost) - Face Feliz', fontsize=14, pad=15)
plt.xlabel('Época (Frames de 1 segundo)', fontsize=12)
plt.ylabel('Acurácia (LOOCV)', fontsize=12)
plt.xticks(frames)
plt.yticks(np.arange(0.3, 1.0, 0.1))
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()

# Salva o gráfico
caminho_grafico = os.path.join(DIR_REPORTS, 'dinamica_temporal_face_feliz.png')
plt.savefig(caminho_grafico, dpi=300)
plt.show()

print("="*80)
print(f"Análise concluída. Gráfico salvo em: {caminho_grafico}")


Iniciando Análise Temporal: XGBoost Frame a Frame (Face Feliz)
Frame 01/30 concluído | Acurácia: 58.14%
Frame 02/30 concluído | Acurácia: 55.81%
Frame 03/30 concluído | Acurácia: 55.81%
Frame 04/30 concluído | Acurácia: 60.47%
Frame 05/30 concluído | Acurácia: 48.84%
Frame 06/30 concluído | Acurácia: 67.44%
Frame 07/30 concluído | Acurácia: 60.47%
Frame 08/30 concluído | Acurácia: 46.51%
Frame 09/30 concluído | Acurácia: 62.79%
Frame 10/30 concluído | Acurácia: 40.48%
Frame 11/30 concluído | Acurácia: 25.00%
Frame 12/30 concluído | Acurácia: 100.00%


ValueError: The groups parameter contains fewer than 2 unique groups (['control_12_faces']). LeaveOneGroupOut expects at least 2.

In [4]:
# 03. AGREGAÇÃO E TESTE DA NOVA FEATURE ERSP (N170)

import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

print("\n" + "="*80)
print("Iniciando Teste Final: Média das Épocas + Feature ERSP (N170)")
print("="*80)

# 1. Carregamento e Agregação (O Cancelamento do Ruído)
df_frames = pd.read_csv('../reports/features_frame_a_frame_ERSP.csv')
df_feliz = df_frames[df_frames['Condicao'] == 'Face Feliz'].copy()

# Agrupa por ID e tira a média de todas as colunas numéricas (Restaurando a SNR do EEG)
colunas_agrupamento = ['ID', 'Grupo']
features_numericas = [c for c in df_feliz.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num']]

df_agrupado = df_feliz.groupby(colunas_agrupamento)[features_numericas].mean().reset_index()

# 2. Preparação para o Machine Learning
df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

X = df_agrupado[features_numericas].values
y = df_agrupado['Target'].values
pacientes = df_agrupado['ID'].values

n_controles = len(df_agrupado[df_agrupado['Target'] == 0])
n_tea = len(df_agrupado[df_agrupado['Target'] == 1])
peso_classes = n_controles / n_tea if n_tea > 0 else 1.0

print(f"Dataset restaurado: {len(df_agrupado)} pacientes ({n_controles} Controles, {n_tea} TEA)")

# 3. Pipeline LOOCV Rigoroso
loo = LeaveOneOut()
previsoes = []
valores_reais = []
importancias_acumuladas = np.zeros(len(features_numericas))

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # A. Escalonamento
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # B. Seleção de Features (ANOVA - As 15 melhores)
    selector = SelectKBest(score_func=f_classif, k=15)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)
    
    # C. Treinamento XGBoost
    modelo = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        scale_pos_weight=peso_classes,
        eval_metric='logloss',
        random_state=42
    )
    modelo.fit(X_train_sel, y_train)
    
    # D. Previsão cega
    y_pred = modelo.predict(X_test_sel)
    previsoes.extend(y_pred)
    valores_reais.extend(y_test)
    
    # E. Rastreamento de Importância
    mask_selecionadas = selector.get_support()
    importancias_parciais = np.zeros(len(features_numericas))
    importancias_parciais[mask_selecionadas] = modelo.feature_importances_
    importancias_acumuladas += importancias_parciais

# 4. Avaliação de Resultados
acc = accuracy_score(valores_reais, previsoes)
cm = confusion_matrix(valores_reais, previsoes)

print(f"\n[RESULTADOS GERAIS]")
print(f"Acurácia Global LOOCV: {acc*100:.2f}%")
print(f"Matriz de Confusão (0=Controle, 1=TEA):\n{cm}")

# 5. Ranking das Features Mais Preditivas
importancias_medias = importancias_acumuladas / len(X)
df_ranking = pd.DataFrame({
    'Feature': features_numericas,
    'Importancia': importancias_medias
}).sort_values(by='Importancia', ascending=False)

print("\n[TOP 10 VARIÁVEIS MAIS IMPORTANTES]")
print(df_ranking[df_ranking['Importancia'] > 0].head(15).to_string(index=False))
print("="*80)


Iniciando Teste Final: Média das Épocas + Feature ERSP (N170)
Dataset restaurado: 42 pacientes (24 Controles, 18 TEA)

[RESULTADOS GERAIS]
Acurácia Global LOOCV: 73.81%
Matriz de Confusão (0=Controle, 1=TEA):
[[19  5]
 [ 6 12]]

[TOP 10 VARIÁVEIS MAIS IMPORTANTES]
     Feature  Importancia
  BetaRel_F7     0.173738
AlphaRel_Fp2     0.159095
 GammaRel_T4     0.115206
  BetaRel_F8     0.112108
 AlphaRel_T5     0.099234
 GammaRel_F7     0.065452
 AlphaRel_P3     0.044717
 AlphaRel_T4     0.039544
 AlphaRel_T3     0.028562
 AlphaRel_F7     0.026716
 BetaRel_Fp2     0.023271
 GammaRel_C4     0.020550
 GammaRel_F8     0.020129
  BetaRel_T4     0.017377
GammaRel_Fp2     0.015830
